## Wheel wavelet decomposition

In [1]:
prefix = '/home/ines/repositories/'
prefix = '/Users/ineslaranjeira/Documents/Repositories/'

In [ ]:
""" 
IMPORTS
"""
import os
import numpy as np
import pandas as pd
from one.api import ONE
from scipy import stats

# Get my functions
from segmentation_functions import idxs_from_files, fast_wavelet_morlet_convolution_parallel
one = ONE(mode='remote')

In [ ]:
""" 
LOAD DATA AND PARAMETERS
"""
# LOAD DATA

data_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/segmentation/design_matrices/'
results_path = '/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/segmentation/wheel_wavelets/'

all_files = os.listdir(data_path)
design_matrices = [item for item in all_files if 'design_matrix' in item and 'standardized' not in item]
idxs, mouse_names = idxs_from_files(design_matrices)

# Wavelet decomposition
f = np.array([.25, .5, 1, 2, 4, 8, 16, 32])
omega0 = 5

In [7]:
# Loop through animals
files = os.listdir(results_path)
sessions_to_process = []

for m, mat in enumerate(idxs):
    mouse_name = mat[37:]
    session = mat[:36]

    """ SAVE DATA """       
    # Save wavelets
    subname = "wheel_vel_wavelets_"
    filename = subname + str(session) + '_'  + mouse_name

    if filename not in files:
        sessions_to_process.append((session))

len(sessions_to_process)

318

In [8]:
sessions_to_exclude = ['2d7c0f7f-e805-404b-914a-23d83998e08e', # bad right cam
'7691eeb3-715b-4571-8fda-6bb57aab8253', # bad view of paws
'bd8b204f-a42e-45c1-a8f0-71c6223a6657', # bad right camera
'f3eeb2d4-87ce-49ae-8a74-21665f6f1536', # moving licks
'650a0a90-4bf3-4489-9bcd-75baf0a49eac', # licks fail
'495bee7e-b58e-42ea-8481-4a1bfedca54a', # timestamps
'1db57661-5ad3-4465-b9ee-08473af9c2e8', # timestamps
'6a369bfa-a70b-4147-af25-d03772ff8d96', # timestamps
'7050ae29-a99e-43f1-aa42-b4416200351c', # timestamps
'3fa080ff-bcce-43e8-bd5f-601f0591f785', # timestamps
'5c454bcb-ae77-42da-a8d2-b6463ea9f21b', # bad licks
'c728f6fd-58e2-448d-aefb-a72c637b604c' # no data can be loaded
]

In [9]:
sessions_to_process = [x for x in sessions_to_process if x not in sessions_to_exclude]

# Wavelets

In [ ]:
concatenated_subsampled = np.array([])
var = ['avg_wheel_vel']

for m, mat in enumerate(sessions_to_process):

    file_path = one.eid2path(mat)
    if prefix == '/home/ines/repositories/':
        mouse_name = file_path.parts[8]
    else:
        mouse_name = file_path.parts[7]

    session = mat
    
    filename = data_path + "design_matrix_" + str(session) + '_'  + mouse_name
    design_matrix = pd.read_parquet(filename)
    
    array = np.array(design_matrix[var[0]]) 
    not_nan = ~np.isnan(array)
    clean_array = array[not_nan]# np.array(stats.zscore(design_matrix[var][not_nan_x]))
    
    # Wavelet decomposition of paw position
    dt = np.round(np.mean(np.diff(design_matrix['Bin'])), 3)
    amp, Q, x_hat = fast_wavelet_morlet_convolution_parallel(clean_array, f, omega0, dt)

    # Wavelet transforms
    for i, frequency in enumerate(f):
        # Create new column with frequency
        design_matrix[str(var[0]+str(frequency))] = np.nan
        design_matrix[str(var[0]+str(frequency))][not_nan] = amp[i, :]

    # """ SAVE DATA """       
    # # Save wavelets
    # subname = "wheel_vel_wavelets_"
    # filename = results_path + subname + str(session) + '_'  + mouse_name
    # design_matrix.to_parquet(filename, compression='gzip')  
    # print(mat)

KeyboardInterrupt: 